# Config A vs Config B — Comparative Analysis

**Purpose:** This notebook compares the best model from Config A (Dynamic Programming — Policy Iteration) against the best model from Config B (PPO) across five lenses: raw performance, training efficiency, treatment strategy, algorithm capabilities, and robustness to clinical noise.

Config A and Config B run on **different environments** — the same clinical task but with different observation representations. A direct survival-rate comparison is therefore not meaningful on its own; the correct metric is **delta above the respective random baseline**, which normalises for environment difficulty.

| Config | Observation | Best algorithm | Environment |
|--------|-------------|----------------|-------------|
| A | Discrete integer (0–715) | Dynamic Programming (Policy Iteration) | Clean, lam=0.02 |
| B | 47-dim continuous vector | PPO | Clinical wrappers + lam=0.02 |


In [ ]:
import sys, os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings; warnings.filterwarnings('ignore')

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir(project_root)

from envs.env_setup import make_sepsis_env, N_STATES, N_ACTIONS, SOFA_BIAS, INTENSITY
from envs.wrappers import make_clinical_env
from utils.evaluation import eval_agent
from agents.config_a.dp import policy_iteration, get_dp_policy
from agents.config_a.q_learning import QLearningAgent, QLearningConfig
from agents.config_a.sarsa import SARSAAgent, SARSAConfig
from agents.config_b.dqn import load_dqn
from agents.config_b.ppo import load_ppo

SEED       = 42
N_EVAL     = 1000
COMP_DIR   = 'results/comparison'
os.makedirs(COMP_DIR, exist_ok=True)
os.makedirs('plots', exist_ok=True)

# ── colour palette shared across all plots ────────────────────────────────
COLORS = {
    'random_a':  '#adb5bd',
    'dp':        'steelblue',
    'ql':        'tab:orange',
    'sarsa':     'tab:green',
    'random_b':  '#ced4da',
    'dqn':       'mediumpurple',
    'ppo':       'tab:red',
}

print('Setup complete.')


---
## 1. Load Models and Evaluate

All evaluations use the fixed protocol: **1,000 episodes, seed=42**.

- **Config A models** are evaluated on `make_sepsis_env()` (discrete, λ=0.02).
- **Config B models** are evaluated twice: on a **clean** environment (no wrappers) and on the **clinical** environment (all three failure wrappers active).

Results are cached to `results/comparison/` so this cell is instant on re-runs.


In [ ]:
CACHE_A   = f'{COMP_DIR}/config_a_evals.npz'
CACHE_B   = f'{COMP_DIR}/config_b_evals.npz'

# ═══════════════════════════════════════════════════════════════════════
# Config A
# ═══════════════════════════════════════════════════════════════════════
if os.path.exists(CACHE_A):
    _a = np.load(CACHE_A, allow_pickle=True)
    a_surv  = _a['surv'].tolist()
    a_ret   = _a['ret'].tolist()
    a_eplen = _a['eplen'].tolist()
    a_labels = list(_a['labels'])
    print('Config A: loaded from cache')
else:
    print('Config A: evaluating all algorithms ...')
    np.random.seed(SEED)

    # Random baseline
    rnd_a   = eval_agent(lambda obs: int(np.random.randint(N_ACTIONS)),
                         make_sepsis_env, N_EVAL, SEED)

    # DP — Policy Iteration (runs in ~2 s)
    _env_raw = make_sepsis_env(verbose=False).unwrapped
    P, R     = _env_raw._tx_mat, _env_raw._r_mat
    pi_pol, _, _ = policy_iteration(P, R, gamma=1.0)
    dp_pi    = eval_agent(get_dp_policy(pi_pol), make_sepsis_env, N_EVAL, SEED)

    # Q-Learning — load cached Q-table
    ql_Q     = np.load('results/config_a/qlearning_qtable.npy')
    with open('results/config_a/qlearning_best_config.json') as f:
        ql_cfg_d = json.load(f)
    ql_agent = QLearningAgent(N_STATES, N_ACTIONS,
                              QLearningConfig(**ql_cfg_d))
    ql_agent.Q = ql_Q
    ql_res   = eval_agent(ql_agent.get_policy(), make_sepsis_env, N_EVAL, SEED)

    # SARSA — load cached Q-table
    sa_Q     = np.load('results/config_a/sarsa_qtable.npy')
    with open('results/config_a/sarsa_best_params.json') as f:
        sa_params = json.load(f)['params']
    sa_agent = SARSAAgent(N_STATES, N_ACTIONS, SARSAConfig(**sa_params))
    sa_agent.Q = sa_Q
    sa_res   = eval_agent(sa_agent.get_policy(), make_sepsis_env, N_EVAL, SEED)

    a_labels = ['Random', 'DP (PI)', 'Q-Learning', 'SARSA']
    a_surv   = [rnd_a['survival_rate'], dp_pi['survival_rate'],
                ql_res['survival_rate'], sa_res['survival_rate']]
    a_ret    = [rnd_a['mean_return'],   dp_pi['mean_return'],
                ql_res['mean_return'],  sa_res['mean_return']]
    a_eplen  = [rnd_a['mean_ep_length'], dp_pi['mean_ep_length'],
                ql_res['mean_ep_length'], sa_res['mean_ep_length']]

    np.savez(CACHE_A, surv=a_surv, ret=a_ret, eplen=a_eplen,
             labels=a_labels)
    print('  Saved to cache.')

print('Config A results:')
for lbl, s in zip(a_labels, a_surv):
    print(f'  {lbl:<15}: {s:.1%}')

# ═══════════════════════════════════════════════════════════════════════
# Config B
# ═══════════════════════════════════════════════════════════════════════
if os.path.exists(CACHE_B):
    _b = np.load(CACHE_B, allow_pickle=True)
    b_surv_clean  = _b['surv_clean'].tolist()
    b_surv_clin   = _b['surv_clin'].tolist()
    b_ret_clean   = _b['ret_clean'].tolist()
    b_eplen_clean = _b['eplen_clean'].tolist()
    b_labels      = list(_b['labels'])
    print('Config B: loaded from cache')
else:
    print('Config B: evaluating DQN and PPO ...')
    dqn_model = load_dqn('results/config_b/dqn_model')
    ppo_model = load_ppo('results/config_b/ppo_model')

    dqn_fn = lambda obs: int(dqn_model.predict(
                np.array(obs, dtype=np.float32), deterministic=True)[0])
    ppo_fn = lambda obs: int(ppo_model.predict(
                np.array(obs, dtype=np.float32), deterministic=True)[0])

    make_clean  = lambda: make_clinical_env(
        malfunction_prob=0.0, missing_prob=0.0, event_prob=0.0)
    make_clinic = make_clinical_env

    rnd_b_c  = eval_agent(lambda obs: int(np.random.randint(N_ACTIONS)),
                          make_clean, N_EVAL, SEED)
    dqn_c    = eval_agent(dqn_fn, make_clean,  N_EVAL, SEED)
    ppo_c    = eval_agent(ppo_fn, make_clean,  N_EVAL, SEED)
    dqn_cl   = eval_agent(dqn_fn, make_clinic, N_EVAL, SEED)
    ppo_cl   = eval_agent(ppo_fn, make_clinic, N_EVAL, SEED)
    rnd_b_cl = eval_agent(lambda obs: int(np.random.randint(N_ACTIONS)),
                          make_clinic, N_EVAL, SEED)

    b_labels      = ['Random (clean)', 'DQN (clean)', 'PPO (clean)',
                     'Random (clin.)', 'DQN (clin.)', 'PPO (clin.)']
    b_surv_clean  = [rnd_b_c['survival_rate'], dqn_c['survival_rate'], ppo_c['survival_rate']]
    b_surv_clin   = [rnd_b_cl['survival_rate'], dqn_cl['survival_rate'], ppo_cl['survival_rate']]
    b_ret_clean   = [rnd_b_c['mean_return'], dqn_c['mean_return'], ppo_c['mean_return']]
    b_eplen_clean = [rnd_b_c['mean_ep_length'], dqn_c['mean_ep_length'], ppo_c['mean_ep_length']]

    np.savez(CACHE_B, surv_clean=b_surv_clean, surv_clin=b_surv_clin,
             ret_clean=b_ret_clean, eplen_clean=b_eplen_clean, labels=b_labels)
    print('  Saved to cache.')

print('\nConfig B results (clean / clinical):')
names_b = ['Random', 'DQN', 'PPO']
for n, sc, scl in zip(names_b, b_surv_clean, b_surv_clin):
    print(f'  {n:<8}: clean={sc:.1%}  clinical={scl:.1%}')


---
## 2. Performance vs Random Baseline

Because Config A and Config B have different random baselines (74.3% vs ~68.3%), comparing absolute survival rates across configs is misleading — the environments differ in difficulty. The fair comparison metric is the **delta above the respective random baseline** (Δ above random).

A positive Δ means the algorithm learned to do better than chance; a negative or zero Δ means it did not.


In [ ]:
# ── Summary table ─────────────────────────────────────────────────────
rnd_a  = a_surv[0]           # Config A random baseline
rnd_bc = b_surv_clean[0]     # Config B random (clean)
rnd_bl = b_surv_clin[0]      # Config B random (clinical)

rows = [
    # Config A
    {'Config': 'A', 'Algorithm': 'Random Baseline',
     'Survival': rnd_a,         'Delta vs Random': 0.0,
     'Mean Return': a_ret[0],   'Env': 'Discrete (clean)'},
    {'Config': 'A', 'Algorithm': 'DP — Policy Iteration',
     'Survival': a_surv[1],     'Delta vs Random': a_surv[1] - rnd_a,
     'Mean Return': a_ret[1],   'Env': 'Discrete (clean)'},
    {'Config': 'A', 'Algorithm': 'Q-Learning',
     'Survival': a_surv[2],     'Delta vs Random': a_surv[2] - rnd_a,
     'Mean Return': a_ret[2],   'Env': 'Discrete (clean)'},
    {'Config': 'A', 'Algorithm': 'SARSA',
     'Survival': a_surv[3],     'Delta vs Random': a_surv[3] - rnd_a,
     'Mean Return': a_ret[3],   'Env': 'Discrete (clean)'},
    # Config B clean
    {'Config': 'B', 'Algorithm': 'Random Baseline',
     'Survival': rnd_bc,        'Delta vs Random': 0.0,
     'Mean Return': b_ret_clean[0], 'Env': 'Continuous (clean)'},
    {'Config': 'B', 'Algorithm': 'DQN (clean env)',
     'Survival': b_surv_clean[1], 'Delta vs Random': b_surv_clean[1] - rnd_bc,
     'Mean Return': b_ret_clean[1], 'Env': 'Continuous (clean)'},
    {'Config': 'B', 'Algorithm': 'PPO (clean env)',
     'Survival': b_surv_clean[2], 'Delta vs Random': b_surv_clean[2] - rnd_bc,
     'Mean Return': b_ret_clean[2], 'Env': 'Continuous (clean)'},
    # Config B clinical
    {'Config': 'B', 'Algorithm': 'DQN (clinical env)',
     'Survival': b_surv_clin[1], 'Delta vs Random': b_surv_clin[1] - rnd_bl,
     'Mean Return': '—',          'Env': 'Continuous (all wrappers)'},
    {'Config': 'B', 'Algorithm': 'PPO (clinical env)',
     'Survival': b_surv_clin[2], 'Delta vs Random': b_surv_clin[2] - rnd_bl,
     'Mean Return': '—',          'Env': 'Continuous (all wrappers)'},
]

df = pd.DataFrame(rows)
df['Survival (%)']     = (df['Survival'] * 100).round(1)
df['Δ above random (pp)'] = (df['Delta vs Random'] * 100).round(1)
df['Mean Return'] = df['Mean Return'].apply(
    lambda x: f'{x:.4f}' if isinstance(x, float) else x)
print(df[['Config','Algorithm','Survival (%)','Δ above random (pp)',
          'Mean Return','Env']].to_string(index=False))

# ── Bar chart: Δ above random ──────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5),
                         gridspec_kw={'width_ratios': [4, 3]})

# --- Left: all algorithms, absolute survival ---
ax = axes[0]
_algos_a  = ['Random', 'DP (PI)', 'Q-Learning', 'SARSA']
_algos_b  = ['Random', 'DQN\n(clean)', 'PPO\n(clean)',
             'DQN\n(clinical)', 'PPO\n(clinical)']
_surv_a   = [s*100 for s in a_surv]
_surv_b   = [rnd_bc*100, b_surv_clean[1]*100, b_surv_clean[2]*100,
             b_surv_clin[1]*100, b_surv_clin[2]*100]
_clrs_a   = [COLORS['random_a'], COLORS['dp'], COLORS['ql'], COLORS['sarsa']]
_clrs_b   = [COLORS['random_b'], COLORS['dqn'], COLORS['ppo'],
             COLORS['dqn'], COLORS['ppo']]

x_a = np.arange(len(_algos_a))
x_b = np.arange(len(_algos_b)) + len(_algos_a) + 0.8

bars_a = ax.bar(x_a, _surv_a, color=_clrs_a, width=0.6, alpha=0.88)
bars_b = ax.bar(x_b, _surv_b, color=_clrs_b, width=0.6, alpha=0.88)

# Add value labels
for bar in list(bars_a) + list(bars_b):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=8)

# Random baselines as dashed lines within each group
ax.hlines(rnd_a*100, x_a[0]-0.4, x_a[-1]+0.4,
          colors='dimgrey', linestyles='--', linewidth=1.5,
          label=f'Random A ({rnd_a*100:.1f}%)')
ax.hlines(rnd_bc*100, x_b[0]-0.4, x_b[-1]+0.4,
          colors='slategrey', linestyles=':', linewidth=1.5,
          label=f'Random B ({rnd_bc*100:.1f}%)')

_all_ticks = list(x_a) + list(x_b)
_all_lbls  = _algos_a + _algos_b
ax.set_xticks(_all_ticks); ax.set_xticklabels(_all_lbls, fontsize=9)
ax.set_ylabel('Survival Rate (%)')
ax.set_title('Survival Rate — All Algorithms', fontsize=12)
ax.axvline(len(_algos_a) + 0.3, color='black', linewidth=0.8, linestyle='-', alpha=0.3)
ax.text(1.5, ax.get_ylim()[0] + 1, 'Config A', ha='center', fontsize=9, color='grey')
ax.text(len(_algos_a)+2.2, ax.get_ylim()[0] + 1, 'Config B', ha='center', fontsize=9, color='grey')
ax.legend(fontsize=8)

# --- Right: delta above random (the fair comparison) ---
ax2 = axes[1]
_d_algos = ['DP (PI)', 'Q-Learning', 'SARSA', 'DQN\nclean', 'PPO\nclean']
_deltas  = [
    (a_surv[1] - rnd_a)*100,
    (a_surv[2] - rnd_a)*100,
    (a_surv[3] - rnd_a)*100,
    (b_surv_clean[1] - rnd_bc)*100,
    (b_surv_clean[2] - rnd_bc)*100,
]
_d_clrs = [COLORS['dp'], COLORS['ql'], COLORS['sarsa'],
           COLORS['dqn'], COLORS['ppo']]

x2 = np.arange(len(_d_algos))
bars2 = ax2.bar(x2, _deltas, color=_d_clrs, width=0.55, alpha=0.88)
ax2.axhline(0, color='black', linewidth=1.0)
for bar, v in zip(bars2, _deltas):
    va = 'bottom' if v >= 0 else 'top'
    offset = 0.1 if v >= 0 else -0.1
    ax2.text(bar.get_x() + bar.get_width()/2, v + offset,
             f'{v:+.1f}pp', ha='center', va=va, fontsize=9, fontweight='bold')
ax2.set_xticks(x2); ax2.set_xticklabels(_d_algos, fontsize=9)
ax2.set_ylabel('Δ above random baseline (pp)')
ax2.set_title('Learning Gain vs Random Baseline\n(clean environments only)', fontsize=11)
ax2.set_ylim(min(_deltas) - 2, max(_deltas) + 2)

plt.tight_layout()
plt.savefig('plots/comparison_performance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: plots/comparison_performance.png')


**Reading the charts:**

- **DP dominates Config A** with +8.9 pp above random. It holds the oracle upper bound because it solves the Bellman equations directly from the full transition model — no samples, no approximation error.

- **Q-Learning and SARSA did not learn to beat random** in Config A. Both sit at or below the random baseline after 30,000 training episodes. The cause is not algorithmic failure but *sample starvation*: with 17,900 Q-table entries and ~300,000 total steps, each entry receives on average only ~17 updates. Combined with sparse binary terminal rewards, credit assignment across ~10-step episodes is unreliable.

- **PPO and DQN do learn in Config B** (+7.7 pp and +6.1 pp over random in the clean environment). Function approximation with neural networks allows generalisation across the continuous 47-dimensional observation space — something a Q-table simply cannot do.

- **The clinical wrapper gap** (clean vs clinical for Config B) is driven almost entirely by `AcuteEventEnv`: a 1% per-step sudden-death probability causes ~9% of episodes to end irreversibly regardless of action taken. This sets a hard ceiling that no algorithm can overcome, bringing both agents back toward the random baseline in the full clinical setting.

- **DP (83.2%) outperforms PPO clean (76.0%)** not because DP is a 'better' algorithm — they operate in entirely different settings. DP has perfect knowledge of the transition model and a clean discrete state; PPO must generalise from noisy 47-dimensional observations in a stochastic environment. The gap illustrates the *cost of model-free learning* and the *cost of a harder observation space*.


---
## 3. Algorithm Capabilities at a Glance

The five algorithms span the full spectrum from model-based to model-free and from tabular to function-approximation methods. The table below summarises their fundamental differences along dimensions relevant to this clinical task.

| Property | DP (PI/VI) | Q-Learning | SARSA | DQN | PPO |
|---|---|---|---|---|---|
| **Config** | A | A | A | B | B |
| **Approach** | Model-based | Model-free | Model-free | Model-free | Model-free |
| **Policy type** | Deterministic | ε-greedy → greedy | ε-greedy → greedy | ε-greedy → greedy | Stochastic → deterministic |
| **On/Off policy** | — | Off-policy | On-policy | Off-policy | On-policy |
| **Requires P(s'\|s,a)?** | ✅ Yes | ❌ No | ❌ No | ❌ No | ❌ No |
| **Handles continuous obs?** | ❌ No | ❌ No | ❌ No | ✅ Yes | ✅ Yes |
| **Robust to obs noise?** | ❌ No | ❌ No | ❌ No | ✅ Yes | ✅ Yes |
| **Training interactions** | 0 (model-based) | ~300k steps | ~300k steps | 100k steps | 100k steps |
| **Exploration mechanism** | None needed | ε-greedy decay | ε-greedy decay | ε-greedy decay | Entropy bonus |
| **δ above random (clean)** | **+8.9 pp** | −0.3 pp | 0.0 pp | +6.1 pp | **+7.7 pp** |

**Key insight:** DP and PPO are the only two algorithms that reliably beat random — but they do so for fundamentally different reasons. DP exploits complete knowledge of the MDP; PPO exploits neural network generalisation. Q-Learning and SARSA fall into a gap: they lack the model knowledge of DP and lack the function approximation of deep RL, making them insufficient for this specific problem.


---
## 4. Learning Efficiency — From Equations to Gradients

DP has no training curve — it computes the optimal policy analytically. For the four model-free algorithms, the learning curve tracks rolling survival rate as training progresses.

Note the **different x-axes**: Config A tracks *episodes*, Config B tracks *timesteps* (since PPO and DQN collect variable-length rollouts rather than discrete episodes).


In [ ]:
# ── Load training histories ────────────────────────────────────────────
ql_returns    = list(np.load('results/config_a/qlearning_returns.npy'))
sarsa_returns = list(np.load('results/config_a/sarsa_returns.npy'))

dqn_cb = np.load('results/config_b/dqn_callback.npz', allow_pickle=True)
ppo_cb = np.load('results/config_b/ppo_callback.npz', allow_pickle=True)

dqn_survivals = dqn_cb['episode_survivals'].astype(float)
dqn_ts        = dqn_cb['episode_timesteps']
ppo_survivals = ppo_cb['episode_survivals'].astype(float)
ppo_ts        = ppo_cb['episode_timesteps']

W = 1000

ql_roll   = np.convolve([r > 0 for r in ql_returns],
                        np.ones(W)/W, mode='valid') * 100
sarsa_roll= np.convolve([r > 0 for r in sarsa_returns],
                        np.ones(W)/W, mode='valid') * 100

dqn_roll  = np.convolve(dqn_survivals, np.ones(W)/W, mode='valid') * 100
dqn_ts_x  = np.cumsum(dqn_ts)[W-1:]
ppo_roll  = np.convolve(ppo_survivals, np.ones(W)/W, mode='valid') * 100
ppo_ts_x  = np.cumsum(ppo_ts)[W-1:]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Config A: episode-based ────────────────────────────────────────────
ax = axes[0]
x_ql   = np.arange(W-1, len(ql_returns))
x_sa   = np.arange(W-1, len(sarsa_returns))
ax.plot(x_ql,  ql_roll,    color=COLORS['ql'],   linewidth=1.5, label='Q-Learning')
ax.plot(x_sa,  sarsa_roll, color=COLORS['sarsa'], linewidth=1.5, label='SARSA')
ax.axhline(rnd_a*100, color=COLORS['random_a'], linestyle='--', linewidth=1.5,
           label=f'Random ({rnd_a*100:.1f}%)')
ax.axhline(a_surv[1]*100, color=COLORS['dp'], linestyle=':', linewidth=2.0,
           label=f'DP oracle ({a_surv[1]*100:.1f}%)')
ax.set_xlabel('Training Episode')
ax.set_ylabel('Rolling Survival Rate (%) [w=1,000]')
ax.set_title('Config A — Q-Learning & SARSA\n(tabular model-free)')
ax.legend(fontsize=9)
ax.set_ylim(rnd_a*100 - 10, a_surv[1]*100 + 5)

# ── Config B: timestep-based ───────────────────────────────────────────
ax2 = axes[1]
ax2.plot(dqn_ts_x, dqn_roll, color=COLORS['dqn'], linewidth=1.5, label='DQN')
ax2.plot(ppo_ts_x, ppo_roll, color=COLORS['ppo'], linewidth=1.5, label='PPO')
ax2.axhline(rnd_bc*100, color=COLORS['random_b'], linestyle='--', linewidth=1.5,
            label=f'Random ({rnd_bc*100:.1f}%)')
ax2.set_xlabel('Training Timestep')
ax2.set_ylabel('Rolling Survival Rate (%) [w=1,000 ep]')
ax2.set_title('Config B — DQN & PPO\n(deep model-free)')
ax2.legend(fontsize=9)

plt.suptitle('Learning Curves: Config A (left) vs Config B (right)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/comparison_learning_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: plots/comparison_learning_curves.png')


**What the learning curves reveal:**

- **Config A (left):** Q-Learning and SARSA never lift off the random baseline across 30,000 episodes. The DP oracle dotted line at 83.2% shows the gap that could have been reached with model knowledge — the curves confirm that tabular model-free methods cannot close it at this training budget. The curves are not slowly converging; they are stuck.

- **Config B (right):** Both DQN and PPO show a clear upward trend from the random baseline across 100,000 timesteps (~10,000 episodes). DQN's curve is noisier due to ε-greedy exploration; PPO's staircase pattern reflects its rollout-based update structure (one update every ~1,000 steps / ~100 episodes). Crucially, **both algorithms are still rising at 100k steps** — more training would likely improve performance further.

- **The contrast is striking:** in the same episode budget (~10k episodes each), Config A's tabular methods stagnate while Config B's deep RL methods show measurable progress. The difference is not effort — it is the capacity of function approximation to generalise from seen states to unseen ones, which a Q-table cannot do.

- **DP requires zero training episodes** — the analytical solution takes ~2 seconds. This is only possible because the full transition model P(s'|s,a) is available. In Config B, there is no discrete state and no finite transition matrix, so this path is closed.


---
## 5. Treatment Strategy: What Did Each Agent Actually Learn?

Survival rates tell us *whether* an agent performs well; action distributions tell us *how* it treats patients. All five algorithms use the same 25-action space (5 vasopressor levels × 5 IV fluid levels), so their action distributions can be compared directly on the same heatmap scale.

Config A policies are rolled out on the discrete environment; Config B policies on the clean continuous environment. Action 0 = (vaso=0, fluid=0) — no treatment.


In [ ]:
CACHE_ACTS = f'{COMP_DIR}/action_counts.npz'

def collect_actions(policy_fn, env_factory, n_episodes=500, seed=SEED):
    env = env_factory()
    env.reset(seed=seed)
    counts = np.zeros(N_ACTIONS, dtype=int)
    for _ in range(n_episodes):
        obs, _ = env.reset()
        done = False
        while not done:
            a = policy_fn(obs)
            counts[a] += 1
            obs, _, term, trunc, _ = env.step(a)
            done = term or trunc
    env.close()
    return counts

if os.path.exists(CACHE_ACTS):
    _ac = np.load(CACHE_ACTS)
    cnt_rnd_a = _ac['rnd_a']; cnt_dp = _ac['dp']
    cnt_ql    = _ac['ql'];    cnt_sa = _ac['sarsa']
    cnt_rnd_b = _ac['rnd_b']; cnt_dqn = _ac['dqn']; cnt_ppo = _ac['ppo']
    print('Action counts: loaded from cache')
else:
    print('Collecting action distributions (~30 s) ...')
    np.random.seed(SEED)
    # Config A
    _env_raw = make_sepsis_env(verbose=False).unwrapped
    pi_pol2, _, _ = policy_iteration(_env_raw._tx_mat, _env_raw._r_mat, gamma=1.0)
    cnt_rnd_a = collect_actions(lambda obs: int(np.random.randint(N_ACTIONS)),
                                 make_sepsis_env)
    cnt_dp    = collect_actions(get_dp_policy(pi_pol2), make_sepsis_env)

    _ql_Q  = np.load('results/config_a/qlearning_qtable.npy')
    with open('results/config_a/qlearning_best_config.json') as f:
        _ql_cfg = json.load(f)
    _ql_agent = QLearningAgent(N_STATES, N_ACTIONS, QLearningConfig(**_ql_cfg))
    _ql_agent.Q = _ql_Q
    cnt_ql = collect_actions(_ql_agent.get_policy(), make_sepsis_env)

    _sa_Q  = np.load('results/config_a/sarsa_qtable.npy')
    with open('results/config_a/sarsa_best_params.json') as f:
        _sa_params = json.load(f)['params']
    _sa_agent = SARSAAgent(N_STATES, N_ACTIONS, SARSAConfig(**_sa_params))
    _sa_agent.Q = _sa_Q
    cnt_sa = collect_actions(_sa_agent.get_policy(), make_sepsis_env)

    # Config B (clean env for a fair comparison)
    _dqn_m = load_dqn('results/config_b/dqn_model')
    _ppo_m = load_ppo('results/config_b/ppo_model')
    _dqn_fn = lambda obs: int(_dqn_m.predict(
                  np.array(obs, dtype=np.float32), deterministic=True)[0])
    _ppo_fn = lambda obs: int(_ppo_m.predict(
                  np.array(obs, dtype=np.float32), deterministic=True)[0])
    make_clean2 = lambda: make_clinical_env(
        malfunction_prob=0.0, missing_prob=0.0, event_prob=0.0)
    cnt_rnd_b = collect_actions(lambda obs: int(np.random.randint(N_ACTIONS)),
                                 make_clean2)
    cnt_dqn   = collect_actions(_dqn_fn, make_clean2)
    cnt_ppo   = collect_actions(_ppo_fn, make_clean2)

    np.savez(CACHE_ACTS, rnd_a=cnt_rnd_a, dp=cnt_dp, ql=cnt_ql, sarsa=cnt_sa,
             rnd_b=cnt_rnd_b, dqn=cnt_dqn, ppo=cnt_ppo)
    print('  Saved to cache.')

# ── Plot: 2 rows × 4 cols heatmaps ────────────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(18, 9))

specs = [
    # Row 0: Config A
    ('Random (A)',     cnt_rnd_a, COLORS['random_a'], 0, 0),
    ('DP — Policy It.', cnt_dp,   COLORS['dp'],       0, 1),
    ('Q-Learning',     cnt_ql,   COLORS['ql'],       0, 2),
    ('SARSA',          cnt_sa,   COLORS['sarsa'],    0, 3),
    # Row 1: Config B (clean)
    ('Random (B)',     cnt_rnd_b, COLORS['random_b'], 1, 0),
    ('DQN (clean)',    cnt_dqn,   COLORS['dqn'],      1, 1),
    ('PPO (clean)',    cnt_ppo,   COLORS['ppo'],      1, 2),
]

# Find global vmax for consistent colour scale within each row
vmax_a = max(cnt_rnd_a.max(), cnt_dp.max(), cnt_ql.max(), cnt_sa.max())
vmax_b = max(cnt_rnd_b.max(), cnt_dqn.max(), cnt_ppo.max())
norms  = {0: vmax_a / cnt_rnd_a.sum(), 1: vmax_b / cnt_rnd_b.sum()}

for title, cnt, color, row, col in specs:
    ax = axes[row][col]
    mat = cnt.reshape(5, 5).astype(float) / cnt.sum()
    vmax_norm = norms[row]
    im = ax.imshow(mat, cmap='YlOrRd', aspect='auto', vmin=0, vmax=vmax_norm)
    ax.set_xticks(range(5)); ax.set_yticks(range(5))
    ax.set_xticklabels([str(i) for i in range(5)], fontsize=8)
    ax.set_yticklabels([str(i) for i in range(5)], fontsize=8)
    ax.set_xlabel('IV Fluid Level', fontsize=8)
    ax.set_ylabel('Vasopressor Level', fontsize=8)
    ax.set_title(title, fontsize=11, color=color, fontweight='bold')
    # Annotate % in each cell
    for r in range(5):
        for c in range(5):
            v = mat[r, c]
            ax.text(c, r, f'{v*100:.1f}', ha='center', va='center',
                    fontsize=7.5,
                    color='white' if v > vmax_norm * 0.55 else 'black')

# Hide the unused 4th cell in row 1
axes[1][3].axis('off')

fig.suptitle(
    'Action Distribution (fraction of steps per treatment combination)\n'
    'Row 0 = Config A (discrete env)   |   Row 1 = Config B (clean continuous env)\n'
    'Rows = vasopressor level (0=none → 4=max) · Cols = IV fluid level (0=none → 4=max)',
    fontsize=10, y=1.02
)
plt.tight_layout()
plt.savefig('plots/comparison_action_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

# Print top action per policy
print('Most frequent action per policy:')
for title, cnt, *_ in specs:
    top = int(np.argmax(cnt))
    vaso, fluid = top // 5, top % 5
    pct = cnt[top] / cnt.sum() * 100
    print(f'  {title:<22}: action {top:2d} '
          f'(vaso={vaso}, fluid={fluid}) — {pct:.1f}% of steps')
print('Saved: plots/comparison_action_distributions.png')


**What the action distributions reveal about clinical strategy:**

- **Random** produces near-uniform heatmaps in both configs — a flat prior across all 25 treatment combinations. It achieves ~74% and ~68% survival simply because most patients in this cohort survive regardless of treatment.

- **DP (Config A)** concentrates mass on specific treatment combinations, with the highest fraction going to action 0 (no vasopressor, no fluid). This is *not* a failure — it is a deliberate clinical strategy. For many stable states the optimal decision is watchful waiting: the λ=0.02 intensity penalty means aggressive treatment causes net harm when it is not needed. DP has learned precisely which states justify intervention and which do not.

- **Q-Learning (Config A)** is more diffuse than DP but shows a modest preference for low-intensity actions. It has learned something — the distribution is not uniform — but the Q-table is too sparsely trained to commit to a clear strategy.

- **SARSA (Config A)** collapses almost entirely to action 0. This is an artefact of zero Q-initialisation: `numpy.argmax` of an all-zero row returns index 0, so any state with insufficient updates defaults to no treatment. The high concentration is not a learned policy — it is an uninitialised Q-table.

- **DQN and PPO (Config B)** show concentrated action distributions that diverge meaningfully from random. Their policies have learned distinct treatment strategies from the continuous observation vector. The fact that both DQN and PPO concentrate on *different* action regions suggests they discovered different but potentially valid treatment policies from the same environment.

- **Comparing DP with PPO**: Both prefer specific treatment combinations over uniform random treatment. DP's strategy is derived analytically from the MDP; PPO's is learned implicitly from ~10,000 patient episodes. The similarity in *structure* (concentrated, non-uniform distributions) despite completely different learning mechanisms is evidence that both agents found genuinely useful clinical patterns.


---
## 6. Robustness to Clinical Noise

Real ICU deployments face observation noise, missing lab values, and sudden irreversible deterioration events. Config B explicitly tests this via the clinical failure wrappers. Config A algorithms operate on a clean discrete state index and have **no mechanism to handle noisy observations**.

This section quantifies the deployment gap for Config B algorithms and discusses the structural limitation of Config A algorithms in real clinical settings.


In [ ]:
CACHE_ROB = f'{COMP_DIR}/robustness_evals.npz'

wrapper_conditions = {
    'No wrappers':   dict(malfunction_prob=0.0,  missing_prob=0.0,  event_prob=0.0),
    'Noisy obs':     dict(malfunction_prob=0.15, missing_prob=0.0,  event_prob=0.0),
    'Missing obs':   dict(malfunction_prob=0.0,  missing_prob=0.15, event_prob=0.0),
    'Acute events':  dict(malfunction_prob=0.0,  missing_prob=0.0,  event_prob=0.01),
    'All (default)': dict(malfunction_prob=0.15, missing_prob=0.15, event_prob=0.01),
}

if os.path.exists(CACHE_ROB):
    _r = np.load(CACHE_ROB, allow_pickle=True)
    rob_dqn = list(_r['dqn'])
    rob_ppo = list(_r['ppo'])
    rob_rnd = list(_r['rnd'])
    print('Robustness data: loaded from cache')
else:
    print('Running robustness evaluation (5 conditions × 3 policies × 500 ep) ...')
    _dqn_m2 = load_dqn('results/config_b/dqn_model')
    _ppo_m2 = load_ppo('results/config_b/ppo_model')
    _dqn_fn2 = lambda obs: int(_dqn_m2.predict(
                   np.array(obs, dtype=np.float32), deterministic=True)[0])
    _ppo_fn2 = lambda obs: int(_ppo_m2.predict(
                   np.array(obs, dtype=np.float32), deterministic=True)[0])
    _rnd_fn2 = lambda obs: int(np.random.randint(N_ACTIONS))

    rob_dqn, rob_ppo, rob_rnd = [], [], []
    for label, kwargs in wrapper_conditions.items():
        factory = lambda kw=kwargs: make_clinical_env(**kw)
        rd = eval_agent(_dqn_fn2, factory, n_eval_episodes=500, seed=SEED)
        rp = eval_agent(_ppo_fn2, factory, n_eval_episodes=500, seed=SEED)
        rr = eval_agent(_rnd_fn2, factory, n_eval_episodes=500, seed=SEED)
        rob_dqn.append(rd['survival_rate'])
        rob_ppo.append(rp['survival_rate'])
        rob_rnd.append(rr['survival_rate'])
        print(f'  {label:<16}: DQN={rd["survival_rate"]:.1%}  '
              f'PPO={rp["survival_rate"]:.1%}  Rnd={rr["survival_rate"]:.1%}')

    np.savez(CACHE_ROB, dqn=rob_dqn, ppo=rob_ppo, rnd=rob_rnd)
    print('  Saved to cache.')

labels_rob = list(wrapper_conditions.keys())
x_rob      = np.arange(len(labels_rob))
width      = 0.28

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(x_rob - width, [v*100 for v in rob_rnd], width, label='Random',
       color=COLORS['random_b'], alpha=0.85)
ax.bar(x_rob,         [v*100 for v in rob_dqn], width, label='DQN',
       color=COLORS['dqn'],      alpha=0.85)
ax.bar(x_rob + width, [v*100 for v in rob_ppo], width, label='PPO',
       color=COLORS['ppo'],      alpha=0.85)

for i, (d, p, r) in enumerate(zip(rob_dqn, rob_ppo, rob_rnd)):
    ax.text(x_rob[i]-width,  d*100+0.2, f'{d*100:.1f}%', ha='center', va='bottom', fontsize=7.5, color=COLORS['dqn'])
    ax.text(x_rob[i],        p*100+0.2, f'{p*100:.1f}%', ha='center', va='bottom', fontsize=7.5, color=COLORS['ppo'])
    ax.text(x_rob[i]+width,  r*100+0.2, f'{r*100:.1f}%', ha='center', va='bottom', fontsize=7.5, color='grey')

ax.set_xticks(x_rob)
ax.set_xticklabels(labels_rob, fontsize=10)
ax.set_ylabel('Survival Rate (%)')
ax.set_title('Config B — Robustness to Clinical Failure Wrappers\n'
             '(500 episodes per condition, seed=42)', fontsize=12)
ax.legend(fontsize=10)
_margin = 3
ax.set_ylim(max(0, min(rob_rnd+rob_dqn+rob_ppo)*100 - _margin),
            min(100, max(rob_dqn+rob_ppo)*100 + _margin))
plt.tight_layout()
plt.savefig('plots/comparison_robustness.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: plots/comparison_robustness.png')


**Robustness findings:**

- **Noisy and missing observations have negligible impact on DQN and PPO.** Both algorithms maintain nearly the same survival rate under `EpisodicNoisyObsEnv` (15% of episodes with Gaussian noise) and `EpisodicMissingObsEnv` (4 of 47 features zeroed). Neural networks generalise across similar observations — a slightly corrupted 47-dim vector still activates similar neurons and produces a similar action. The 47-dimensional feature space carries substantial redundancy, so removing 4 features leaves the decision largely intact.

- **Acute events are the dominant failure mode and are irreducible.** With 1% sudden death probability per step and ~10 steps per episode, roughly 9% of patients die regardless of action. This is an irreversible environmental event — no learning can prevent it. The result: `AcuteEventEnv` alone (no other wrappers) produces nearly the same survival as the full clinical environment, confirming it as the binding constraint.

- **Config A algorithms cannot be deployed clinically without significant modification.** DP, Q-Learning, and SARSA require an exact discrete state index (0–715). In the real clinical environment, the agent receives a noisy 47-dimensional continuous vector. There is no reliable mapping from this vector back to a discrete state index — a single corrupted physiological measurement would lookup the wrong row in the Q-table. Deep RL methods sidestep this entirely: their neural network input is the raw observation vector, so noise is handled implicitly.

- **PPO is marginally more robust than DQN** across all wrapper conditions. Its on-policy training means the policy is trained directly on the distribution it will encounter at test time — including episodes drawn from the noisy wrapper distribution. DQN's replay buffer may contain cleaner experience from early training, slightly misaligning its Q-network with the current environment distribution.


---
## 7. Summary and Key Takeaways

### What this comparison shows

| Question | Answer |
|---|---|
| Did any agent learn to beat random? | Yes: DP (+8.9 pp), DQN clean (+6.1 pp), PPO clean (+7.7 pp) |
| Did tabular model-free methods learn? | No — Q-Learning and SARSA tied or fell below random in 30k episodes |
| Why does DP outperform PPO? | DP has full MDP access and a clean discrete state; PPO must generalise from noisy continuous observations |
| Why does deep RL succeed where tabular fails? | Neural networks generalise across the continuous 47-dim state space; Q-tables cannot |
| Are the agents clinically robust? | DQN and PPO are robust to noise and missing values; Config A methods are not deployable under clinical noise |
| What is the performance ceiling in Config B? | ~91% survival before irreducible acute events; ~76% (clean) is near the practical ceiling at 100k steps |

### The progression from Config A to Config B

The project traces a natural progression in RL complexity:

1. **DP (Config A)** — the oracle. Shows what is achievable with full model knowledge. Survial: 83.2%. No training required.

2. **Tabular model-free (Config A)** — proves the limitation. Without the model, 30,000 episodes and sparse binary rewards are insufficient to populate 17,900 Q-table entries reliably. Result: tied with random.

3. **Deep RL (Config B)** — closes the gap. With function approximation, the agent generalises across an effectively infinite state space from only 100,000 timesteps. PPO reaches +7.7 pp above random in the clean environment. The remaining gap to DP reflects both the harder observation space and the absence of model knowledge.

### Clinical takeaway

Both DP and PPO have learned **non-trivial treatment strategies** that diverge from random — their action distributions are concentrated and clinically interpretable (watchful waiting for stable patients, targeted treatment for deteriorating ones). The clinical wrapper sensitivity analysis shows that the near-random survival rates in the full clinical environment are driven by irreducible stochastic death events, not by the quality of the learned treatment policy.

A deployed RL agent in an ICU would face the acute event ceiling regardless of algorithm quality. The right evaluation question is not *absolute survival rate* but *survival rate relative to the best achievable under the same environmental constraints* — and by that measure, PPO demonstrates clear and meaningful learning.
